# Programme 2 - Signature, cryptogramme & pages d'examen
- **Signature** (ligne 15) : detection de la presence d'encre dans la zone (bas niveau) ; l'authentification fine contre la base `STUDENT_CLASS_SIGNATURES` est un point d'entree ML.
- **Cryptogramme** (ligne 18) : on compare le petit graphique du bas de page entre toutes les pages par **correlation croisee normalisee (NCC)** sur des signatures binaires.
- **Pages d'examen** : on segmente les **blocs question** (grands rectangles) pour numeroter, puis on lit les cases QCM de la colonne de gauche (cochee via l'encre interieure). Les questions numeriques (mantisse/exposant) relevent de la reconnaissance manuscrite (ML).

In [ ]:
# --- Choix du dossier, du PDF et de la page ------------------------------
# dossier    : "FORM1" / "FORM2" / "FORM3"
# page_index : 0 = page d'identification (PAGE-01) ; 4 = 1re page de questions
import os, glob, numpy as np
import reader_pdf as R

dossier    = "FORM1"
page_index = 4

pdfs = sorted(glob.glob(os.path.join(dossier, "*.pdf")))
if not pdfs:
    raise FileNotFoundError(
        f"Aucun PDF dans '{os.path.abspath(dossier)}'. "
        f"Lance Jupyter depuis la racine du projet, ou mets un chemin complet.")
pdf_path = pdfs[0]
print("PDF utilise :", pdf_path, "| total trouves :", len(pdfs))

pages = R.pdf_to_pages(pdf_path, dpi=200)   # 1 image par page (pdf2image -> fitz en secours)
im    = pages[page_index]
print("nb pages :", len(pages), "| page", page_index, "shape", im.shape)


In [ ]:
import matplotlib.pyplot as plt
# Lecture des blocs question de CETTE page (page de questions)
for k, blk in enumerate(R.read_exam_page(im), start=1):
    print(f"bloc {k}: {blk}")

## Cryptogramme : identique sur toutes les pages ?

In [ ]:
sigs = [R.crypto_sig(p, (0.235, 0.915, 0.07, 0.06)) for p in pages]
print("Cryptogrammes identiques :", R.cryptograms_identical(sigs, thr=0.60))

## Signature (page 1)

In [ ]:
g1 = R.norm_gray(pages[0]); H,W = g1.shape
x0,x1,y0,y1 = 0.02,0.45,0.27,0.46
sig_zone = g1[int(y0*H):int(y1*H), int(x0*W):int(x1*W)]
present = int((sig_zone<0.5).mean() > 0.01)
plt.imshow(sig_zone, cmap='gray'); plt.title(f"Zone signature - presence={present}"); plt.axis('off'); plt.show()